In [0]:
%sql
select * from primeinsurance_analytics.bronze.bronze_cars;

In [0]:
%sql
select * from primeinsurance_analytics.bronze.bronze_claims;

In [0]:
%sql
select * from primeinsurance_analytics.bronze.bronze_customers

In [0]:
%sql
select * from primeinsurance_analytics.bronze.bronze_policy

In [0]:
%sql
select * from primeinsurance_analytics.bronze.bronze_sales

In [0]:
%sql
select * from primeinsurance_analytics.silver.silver_cars

In [0]:
%sql
select * from primeinsurance_analytics.silver.silver_claims;

In [0]:
%sql
select * from primeinsurance_analytics.silver.silver_customers;

In [0]:
%sql
select * from primeinsurance_analytics.silver.silver_policy

In [0]:
%sql
select * from primeinsurance_analytics.silver.silver_sales

In [0]:
%sql
select * from primeinsurance_analytics.silver.quarantine_cars

In [0]:
%sql
select * from primeinsurance_analytics.silver.quarantine_claims

In [0]:
%sql
select * from primeinsurance_analytics.silver.quarantine_customers

In [0]:
%sql
select * from primeinsurance_analytics.silver.quarantine_policy;

In [0]:
%sql
select * from primeinsurance_analytics.silver.quarantine_sales

In [0]:
%sql
select * from primeinsurance_analytics.silver.silver_quality_log

In [0]:
%sql
select * from primeinsurance_analytics.gold.dim_car

In [0]:
%sql
select * from primeinsurance_analytics.gold.dim_customer

In [0]:
%sql
select * from primeinsurance_analytics.gold.dim_policy

In [0]:
%sql
select * from primeinsurance_analytics.gold.fact_claims

In [0]:
%sql
select * from primeinsurance_analytics.gold.fact_sales

In [0]:
%sql
select * from primeinsurance_analytics.gold.agg_claim_processing_time_by_severity

In [0]:
%sql
select * from primeinsurance_analytics.gold.agg_claim_rejection_by_region_month

In [0]:
%sql
select * from primeinsurance_analytics.gold.agg_claim_value_by_region_policy

In [0]:
%sql
select * from primeinsurance_analytics.gold.agg_customer_count_by_region

In [0]:
%sql
select * from primeinsurance_analytics.gold.agg_unsold_inventory_by_model_region

In [0]:
%sql
select * from primeinsurance_analytics.gold.dq_explanation_report

In [0]:
%sql
select * from primeinsurance_analytics.gold.claim_anomaly_explanations

In [0]:
%sql
select * from primeinsurance_analytics.gold.ai_business_insights

In [0]:
%sql
select * from primeinsurance_analytics.gold.rag_query_history

In [0]:
%sql
SHOW GRANTS ON VOLUME primeinsurance_analytics.source_files.raw_files;

In [0]:
%sql
-- Records that PASSED into silver
SELECT 'customers' AS entity, COUNT(*) AS passed_records FROM primeinsurance_analytics.silver.silver_customers
UNION ALL
SELECT 'cars',     COUNT(*) FROM primeinsurance_analytics.silver.silver_cars
UNION ALL
SELECT 'policy',   COUNT(*) FROM primeinsurance_analytics.silver.silver_policy
UNION ALL
SELECT 'sales',    COUNT(*) FROM primeinsurance_analytics.silver.silver_sales
UNION ALL
SELECT 'claims',   COUNT(*) FROM primeinsurance_analytics.silver.silver_claims;

In [0]:
%sql
-- Records that were FLAGGED (quarantined)
SELECT entity, quarantine_reason, quarantine_type, failed_record_count
FROM primeinsurance_analytics.silver.silver_quality_log
ORDER BY entity, failed_record_count DESC;

In [0]:
from pyspark.sql import functions as F

CATALOG = "primeinsurance_analytics"
BRONZE  = f"{CATALOG}.bronze"
SILVER  = f"{CATALOG}.silver"

# =============================================================================
# ENTITY 1 — CUSTOMERS
# =============================================================================

print("CUSTOMERS — BEFORE")
spark.sql(f"""
    SELECT
        COALESCE(customerid, customer_id, cust_id)      AS customer_id_raw,
        COALESCE(reg, region)                           AS region_raw,
        COALESCE(education, edu)                        AS education_raw,
        job                                             AS job_raw,
        COALESCE(marital_status, marital)               AS marital_raw,
        CAST(balance AS DECIMAL(10,2))                  AS balance
    FROM {BRONZE}.bronze_customers
    LIMIT 10
""").show(truncate=False)

print("CUSTOMERS — AFTER")
spark.sql(f"""
    SELECT
        customer_id,
        region,
        education,
        job,
        marital_status,
        balance,
        balance_negative_flag
    FROM {SILVER}.silver_customers
    LIMIT 10
""").show(truncate=False)


# =============================================================================
# ENTITY 2 — CARS
# =============================================================================

print("CARS — BEFORE")
spark.sql(f"""
    SELECT
        car_id,
        name            AS car_name_raw,
        fuel            AS fuel_raw,
        transmission    AS transmission_raw,
        seats           AS seats_raw,
        km_driven       AS km_driven_raw,
        mileage         AS mileage_raw,
        engine          AS engine_raw,
        max_power       AS max_power_raw,
        torque          AS torque_raw
    FROM {BRONZE}.bronze_cars
    LIMIT 10
""").show(truncate=False)

print("CARS — AFTER")
spark.sql(f"""
    SELECT
        car_id,
        car_name,
        fuel,
        transmission,
        seats,
        km_driven,
        mileage_kmpl,
        engine_cc,
        max_power_bhp,
        torque_nm
    FROM {SILVER}.silver_cars
    LIMIT 10
""").show(truncate=False)


# =============================================================================
# ENTITY 3 — POLICY
# =============================================================================

print("POLICY — BEFORE")
spark.sql(f"""
    SELECT
        policy_number,
        policy_bind_date        AS bind_date_raw,
        policy_state            AS state_raw,
        policy_csl              AS csl_raw,
        policy_deductable       AS deductible_raw,
        policy_annual_premium   AS premium_raw,
        umbrella_limit          AS umbrella_raw
    FROM {BRONZE}.bronze_policy
    LIMIT 10
""").show(truncate=False)

print("POLICY — AFTER")
spark.sql(f"""
    SELECT
        policy_number,
        policy_bind_date,
        policy_state,
        policy_csl,
        csl_bodily_injury_per_person,
        csl_bodily_injury_per_accident,
        deductible,
        annual_premium,
        umbrella_limit,
        umbrella_limit_warning,
        is_active
    FROM {SILVER}.silver_policy
    LIMIT 10
""").show(truncate=False)


# =============================================================================
# ENTITY 4 — SALES
# =============================================================================

print("SALES — BEFORE")
spark.sql(f"""
    SELECT
        sales_id,
        ad_placed_on                AS ad_placed_raw,
        sold_on                     AS sold_on_raw,
        original_selling_price      AS price_raw,
        region                      AS region_raw,
        seller_type                 AS seller_type_raw,
        owner                       AS owner_raw
    FROM {BRONZE}.bronze_sales
    LIMIT 10
""").show(truncate=False)

print("SALES — AFTER")
spark.sql(f"""
    SELECT
        sales_id,
        ad_placed_on,
        sold_on,
        days_to_sell,
        selling_price,
        region,
        seller_type,
        owner,
        sale_status,
        is_sold
    FROM {SILVER}.silver_sales
    LIMIT 10
""").show(truncate=False)


# =============================================================================
# ENTITY 5 — CLAIMS
# =============================================================================

print("CLAIMS — BEFORE")
spark.sql(f"""
    SELECT
        claimid                     AS claim_id_raw,
        policyid                    AS policy_id_raw,
        incident_date               AS incident_date_raw,
        claim_logged_on             AS logged_raw,
        claim_processed_on          AS processed_raw,
        claim_rejected              AS rejected_raw,
        collision_type              AS collision_raw,
        police_report_available     AS police_raw,
        property_damage             AS property_dmg_raw,
        injury, property, vehicle
    FROM {BRONZE}.bronze_claims
    LIMIT 10
""").show(truncate=False)

print("CLAIMS — AFTER")
spark.sql(f"""
    SELECT
        claim_id,
        policy_id,
        incident_date,
        claim_logged_on,
        claim_processed_on,
        days_to_process,
        claim_status,
        is_rejected,
        collision_type,
        police_report_available,
        property_damage,
        injury_amount,
        property_amount,
        vehicle_amount,
        total_claim_amount
    FROM {SILVER}.silver_claims
    LIMIT 10
""").show(truncate=False)